## Model of "ISAAC: A Convolutional Neural Network Accelerator with In-Situ Analog Arithmetic in Crossbars", ISCA 2016

Paper by Ali Shafiee, Anirban Nag, Naveen Muralimanohar, Rajeev Balasubramonian, John
Paul Strachan, Miao Hu, R. Stanley Williams, and Vivek Srikumar.

### Description of ISAAC

ISAAC is a ReRAM-based analog CiM accelerator. It explores several concepts in CiM
acceleration, including storing different layers in different arrays and pipelining
inputs/outputs between.

### Tile Level

Twelve macros (called IMAs in the paper) are organized into a tile. Each tile includes a
64kB eDRAM buffer storing 16b inputs/outputs and quantization circuits. The original
paper included sigmoid units at this level, but we replaced them with quantization
circuits to match the other works. ISAAC uses 16b fixed-point quantization for all
operands.

- *Input Path*: Inputs are stored in the eDRAM. An inter-macro network sends inputs to
  macros in the tile.
- *Weight Path*: Weights are kept static in inference and do not move through this
  level.
- *Output Path*: Outputs are gathered from macros via the inter-tile network. They are
  quantized before being stored in the eDRAM.

Next, there are 12 macros in each tile. Inputs and outputs are unicast between macros.

### Macro Level

Eight arrays are organized into a macro with an input register and output register. An
input network sends input vectors to arrays.

The eight arrays can process up to 8x128 = 1024 inputs across all rows, so the input
register is sized 2kB (2B per input). The output register is sized 256B (2B per output,
128 outputs total (8 arrays x 128 columns x 2b per column / 16b per output)). While the
paper does not do this, we double output buffer size to account for higher-precision
accumulation that is important for lower-precision quantization.

- *Input Path*: Inputs are stored in the input buffer and multicast between arrays.
- *Weight Path*: Weights are kept static in inference and do not move through this
  level.
- *Output Path*: Outputs are stored in the output buffer and spatially reduced between
  arrays. Before the output buffer, a shift+add circuit accumulates outputs and corrects
  for offsets caused by slicing.

Next, there are 8 arrays in each macro. Inputs and outputs can be spatially reused
across arrays.

### Array Level

Arrays consist of 128 x 128 ReRAMs. Each array is programmed with weights from one DNN
layer, and each weight filter uses 8 array columns (16b weights, 2b per column). 1-bit
DACs encode inputs across 16 cycles and 8-bit ADCs convert outputs from each column.

Inputs and weights are both assumed to be 16b unsigned fixed-point numbers. Signed
inputs and weights are converted by adding a bias to the inputs and weights.

We note that the original ISAAC paper included a contribution to decrease required ADC
precision. Instead of supporting between 0 and the maximum output of a column, ISAAC
supported only half of the range by offsetting the weight encoding. The two's complement
value was encoded with an offset of half of the maximum representable value, so the
minimum possible output was half of the maximum. This cut the required range (and
therefore precision) of the ADC by one bit. However, this technique relies on a specific
weight encoding that is not compatible with higher-precision quantization. For this
reason, we don't model this technique in our ISAAC model.

- *Input Path*: Inputs pass through 1-bit DACs and appear on the rows of the array.
- *Weight Path*: Weights are stored in the array and are not moved during inference.
- *Output Path*: Outputs are read from the columns of the array with 8-bit ADCs.

Next, there are 128 columns in each array. Inputs are reused between columns.

### Column Level

Each column consists of 128 ReRAM devices. Columns store 2b weight slices.

- *Input Path*: Each input is passed directly to a row in the column.
- *Weight Path*: Weights are not moved during inference.
- *Output Path*: Outputs pass through a current mirror to buffer their values before
  exiting the column.

### Row Level

Each row in a column has one ReRAM device which stores an offset-encoded 2b weight
slice.

- *Input Path*: The input is used for a MAC operation.
- *Weight Path*: A 2b weight is stored in the ReRAM device and is used for a MAC
  operation.
- *Output Path*: The output is supplied by a MAC operation.

In [ ]:
from _scripts import *

display_important_variables("isaac_isca_2016")
get_spec("isaac_isca_2016").arch

### Energy Breakdown

This test explores the energy, area, and latency of the accelerator computing MVM
operations. We note a few differences from the original ISAAC paper. Notably, we made a
few changes to the quantization, and we use data-value-dependent models while ISAAC used
a simple fixed-power model.

We note:
- Energy is dominated by the ADC and memory cells due to the high ADC precision and
    large number of slices.
- Area is dominated by ADC.

In [ ]:
result = get_spec("isaac_isca_2016", add_dummy_main_memory=True).map_workload_to_arch(
    print_progress=False
)
energy_fj = {
    k: v * 1e15
    for k, v in result.per_compute().energy(per_component=True).items()
    if v > 0
}

fig, ax = plt.subplots(figsize=(10, 5))
bar(energy_fj, "Component", "Energy (fJ/MAC)", "ISAAC Energy Breakdown", ax)
plt.tight_layout()
plt.show()

for k, v in sorted(energy_fj.items(), key=lambda x: -x[1]):
    print(f"{k}: {v:.2f} fJ/MAC")

tops = 2 / result.per_compute().latency() / 1e12
tops_w = 2 / result.per_compute().energy() / 1e12
print(f"\nTOPS: {tops:.4f}")
print(f"TOPS/W: {tops_w:.1f}")

### Area Breakdown

Area breakdown is similar to the energy breakdown. ISAAC area is dominated by the ADC.

In [ ]:
area = get_area_breakdown("isaac_isca_2016")
area_um2 = {k: v * 1e12 for k, v in area.items() if v > 0}

fig, ax = plt.subplots(figsize=(10, 5))
bar(area_um2, "Component", "Area (um\u00b2)", "ISAAC Area Breakdown", ax)
plt.tight_layout()
plt.show()

total = sum(area_um2.values())
for k, v in sorted(area_um2.items(), key=lambda x: -x[1]):
    print(f"{k}: {v:.1f} um\u00b2 ({v/total*100:.1f}%)")
print(f"Total: {total:.1f} um\u00b2 ({total/1e6:.4f} mm\u00b2)")

### Running a Full DNN on the Accelerator

In this test, we look at the full-system energy breakdown when running DNNs on the
accelerator.

In [ ]:
woarkload_path = af.examples.workloads.compute_in_memory.alexnet
workload = af.Workload.from_yaml(
    woarkload_path, top_key="workload", jinja_parse_data={"BATCH_SIZE": 64}
)
renames = af.Renames.from_yaml(
    woarkload_path, top_key="renames", jinja_parse_data={"BATCH_SIZE": 64}
)

spec = get_spec("isaac_isca_2016", add_dummy_main_memory=True, n_macros=1)
spec.workload = workload
spec.workload.einsums = spec.workload.einsums
spec.renames = renames
spec.mapper.metrics = af.Metrics.ENERGY

# It's OK to explore loop orders, but will make the search take longer, and is not very
# helpful for CiM accelerators. It's not very helpful because loop orders help us
# capture sliding window reuse, and the CiM macros consume very big tiles at once,
# giving limited benefit from the small overlap between each tile.
spec.mapper.explore_loop_orders = False

result = spec.map_workload_to_arch()
result = result.drop_components_with_zero_energy_and_latency()

In [5]:
display_dnn_results(result, title="AlexNet on ISAAC")

### Visualize the Chosen Mapping

In [ ]:
result